In [6]:
import sys
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"                  # no usar el sistema de descarga que se colgaba
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"     # ocultar el aviso de symlinks de Windows

from pathlib import Path

# --- Añadir la carpeta del proyecto (..) a las rutas de Python para poder importar 'ner' ---
PROJECT_DIR = Path('..').resolve()
sys.path.insert(0, str(PROJECT_DIR))

import torch, gliner, pymupdf
from ner.evaluation import ENTITY_LABELS

print('Python:', sys.version.split()[0])                               # debe ser 3.13.x
print('Project folder:', PROJECT_DIR)                                   # debe acabar en Practica2
print('torch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print('pymupdf:', pymupdf.__version__)
print('Labels:', ENTITY_LABELS)

Python: 3.13.18
Project folder: C:\Users\alexa\Documents\2º DAM\IA\Practica2
torch: 2.15.0+cpu | GPU: False
pymupdf: 2.1
Labels: ('PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS', 'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION')


# 00 - Exploration (GLiNER + PyMuPDF)
**Members:** Alexander Mansilla Arteaga, (partner)

## 1. GLiNER — Basic Usage

In [7]:
from gliner import GLiNER

# --- Usar GPU si existe, si no CPU ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# --- Cargar el modelo pequeño ---
model = GLiNER.from_pretrained("gliner-community/gliner_small-v2.5").to(device)

# --- Texto y etiquetas del ejemplo de GitHub ---
text = """Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese
professional footballer who plays as a forward for and captains both Saudi Pro League
club Al Nassr and the Portugal national team."""
labels = ["Person", "Award", "Date", "Competitions", "Teams"]

# --- Predecir: threshold = confianza mínima (0 a 1) para aceptar una entidad ---
entities = model.predict_entities(text, labels, threshold=0.5)
for entity in entities:
    print(entity["text"], "=>", entity["label"])

Using device: cpu


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Cristiano Ronaldo dos Santos Aveiro => Person
5 February 1985 => Date
Saudi Pro League => Competitions
Al Nassr => Teams
Portugal national team => Teams


In [8]:
# --- Una LISTA de DICCIONARIOS, uno por entidad ---
print(type(entities), len(entities))
entities[0]

<class 'list'> 5


{'start': 0,
 'end': 35,
 'text': 'Cristiano Ronaldo dos Santos Aveiro',
 'label': 'Person',
 'score': 0.986089825630188}

In [9]:
# --- Recortar con start:end y comparar con el campo 'text' ---
for e in entities:
    trozo = text[e['start']:e['end']]
    print(e['start'], e['end'], repr(trozo), trozo == e['text'])

0 35 'Cristiano Ronaldo dos Santos Aveiro' True
42 57 '5 February 1985' True
144 160 'Saudi Pro League' True
166 174 'Al Nassr' True
183 205 'Portugal national team' True


**What does `predict_entities` return?**

It returns a list of dictionaries, one for each detected entity. In our example it found 5 entities.
Each dictionary has:

- `start` / `end`: character positions of the entity in the original text (`start` included,
  `end` excluded, like a Python slice). For example, *Cristiano Ronaldo dos Santos Aveiro* goes from 0 to 35.
- `text`: the substring of the text between `start` and `end`.
- `label`: the entity type assigned by the model (one of the labels we gave it).
- `score`: the confidence of the model (0–1). Only entities with score ≥ threshold (0.5) are returned.

**What are the spans?**

A span is the pair (`start`, `end`): the exact location of the entity in the text.
We checked that `text[start:end] == entity['text']` is `True` for all entities.

Spans matter for evaluation: the strict evaluation compares (start, end, label), so an entity
with the right text but a span shifted by one character counts as an error.

The label *Award* was not predicted because the text does not mention any award.

## 2. GLiNER with our documents and labels


In [10]:
from ner.utils import load_jsonl

VALIDATION_PATH = '../data/plain/validation.jsonl'
samples = load_jsonl(VALIDATION_PATH)

print('Documents:', len(samples))
print('Keys:', list(samples[0].keys()))
print('Text length:', len(samples[0]['text']), 'characters')
print('\n', samples[0]['text'][:600])

print('\nGold entities of the first document:')
for e in samples[0]['entities'][:10]:
    print(f"  {e['label']:<20} {e['start']:>5}-{e['end']:<5} {samples[0]['text'][e['start']:e['end']]}")

Documents: 50
Keys: ['language', 'text', 'entities', 'id']
Text length: 4091 characters

 Servicio de Procedimientos Públicos de la Región Central informa de que Patricia Giménez Domínguez, con documento de identidad 12082063W y domicilio habitual en Avenida de Aragón, 102, 2-B, 17001 Girona, presentó el 2018-04-09 una solicitud relacionada con la inscripción en un registro administrativo sectorial. La petición se registró al sistema con el número ASS-2018-43226, y el justificante de entrada fue remitido a Patricia Giménez Domínguez mediante el canal señalado en el formulario. En la documentación también figuraba Tomás Romero Domínguez, identificado con 12082200R, como representant

Gold entities of the first document:
  ORGANIZATION             0-56    Servicio de Procedimientos Públicos de la Región Central
  PERSON_NAME             72-98    Patricia Giménez Domínguez
  ID_CARD_NUMBER         127-136   12082063W
  ADDRESS                161-202   Avenida de Aragón, 102, 2-B, 17001 Gi

In [11]:
LABELS = list(ENTITY_LABELS)
fragment = samples[0]['text'][:600]

# --- Lo que predice el modelo ---
print('PREDICTED:')
for e in model.predict_entities(fragment, LABELS, threshold=0.5):
    print(f"  {e['label']:<20} {e['score']:.2f}  {e['text']}")

# --- El gold del mismo fragmento (entidades que acaban antes del carácter 600) ---
print('\nGOLD:')
for e in samples[0]['entities']:
    if e['end'] <= 600:
        print(f"  {e['label']:<20}       {fragment[e['start']:e['end']]}")

PREDICTED:
  ORGANIZATION         0.75  Servicio de Procedimientos Públicos
  ORGANIZATION         0.98  Región Central
  PERSON_NAME          0.99  Patricia Giménez Domínguez
  ID_CARD_NUMBER       0.73  12082063W
  ADDRESS              0.93  Avenida de Aragón
  ADDRESS              0.86  17001 Girona
  DATE                 0.98  2018-04-09
  REGISTRATION_NUMBER  0.86  ASS-2018-43226
  PERSON_NAME          0.97  Patricia Giménez Domínguez
  PERSON_NAME          0.97  Tomás Romero Domínguez
  ID_CARD_NUMBER       0.73  12082200R

GOLD:
  ORGANIZATION               Servicio de Procedimientos Públicos de la Región Central
  PERSON_NAME                Patricia Giménez Domínguez
  ID_CARD_NUMBER             12082063W
  ADDRESS                    Avenida de Aragón, 102, 2-B, 17001 Girona
  DATE                       2018-04-09
  REGISTRATION_NUMBER        ASS-2018-43226
  PERSON_NAME                Patricia Giménez Domínguez
  PERSON_NAME                Tomás Romero Domínguez
  ID_CARD_NUMB

**Observations:**

- **Very good results** for `PERSON_NAME`, `DATE` and `REGISTRATION_NUMBER`: all of them are found
  with the exact text and high confidence (0.86–0.99).
- `ID_CARD_NUMBER` is also correct, but with a lower score (0.73), so a higher threshold could miss it.
- **ADDRESS is split into pieces.** The gold address is one entity
  (*Avenida de Aragón, 102, 2-B, 17001 Girona*), but the model returns *Avenida de Aragón* and
  *17001 Girona* separately, and the part *102, 2-B* is not detected at all.
- **ORGANIZATION is also split.** The gold entity is *Servicio de Procedimientos Públicos de la Región
  Central*, but the model returns two entities: *Servicio de Procedimientos Públicos* and *Región Central*.
- Conclusion: the model locates the entities correctly, but it has problems with **long, multi-part
  entities**. In the strict evaluation each piece will count as a false positive and the full gold
  entity as a false negative, so ADDRESS and ORGANIZATION will probably get the worst scores.
  A possible solution is to merge adjacent pieces with the same label (post-processing).

## 3. PyMuPDF — Quick Start

In [12]:
import pymupdf

# --- Abrir el primer PDF limpio y ver su información ---
doc = pymupdf.open('../data/pdf_clean/test-0001.pdf')
print('Pages:', doc.page_count)
print('Metadata:', doc.metadata)

# --- Extraer el texto de todas las páginas ---
pdf_text = ''.join(page.get_text() for page in doc)
doc.close()

print('Characters extracted:', len(pdf_text))
print(pdf_text[:600])

# --- ¿Es idéntico al texto del jsonl? ---
print('\nIdentical to validation.jsonl text?', pdf_text == samples[0]['text'])

Pages: 1
Metadata: {'format': 'PDF 1.3', 'title': 'untitled', 'author': 'anonymous', 'subject': 'unspecified', 'keywords': '', 'creator': 'anonymous', 'producer': 'ReportLab PDF Library - (opensource)', 'creationDate': "D:20260821113855+02'00'", 'modDate': "D:20260821113855+02'00'", 'trapped': '', 'encryption': None}
Characters extracted: 4090
Servicio de Procedimientos Públicos de la Región Central informa de que Patricia Giménez
Domínguez, con documento de identidad 12082063W y domicilio habitual en Avenida de Aragón,
102, 2-B, 17001 Girona, presentó el 2018-04-09 una solicitud relacionada con la inscripción en un
registro administrativo sectorial. La petición se registró al sistema con el número ASS-2018-43226, y
el justificante de entrada fue remitido a Patricia Giménez Domínguez mediante el canal señalado en
el formulario. En la documentación también figuraba Tomás Romero Domínguez, identificado con
12082200R, como representant

Identical to validation.jsonl text? False


In [13]:
# --- Abrir el primer PDF escaneado (fíjate en el sufijo _scanned) ---
doc = pymupdf.open('../data/pdf_scanned/test-0001_scanned.pdf')
print('Pages:', doc.page_count)
print('Images in page 1:', len(doc[0].get_images()))

scanned_text = ''.join(page.get_text() for page in doc)
doc.close()

print('Characters extracted:', len(scanned_text))
print(repr(scanned_text[:300]))

Pages: 1
Images in page 1: 1
Characters extracted: 0
''


**Observations:**

- **Clean PDF:** the PDF has 1 page and was generated with ReportLab. PyMuPDF extracts 4,090
  characters with `page.get_text()`, almost the same text as in `validation.jsonl`, but it is not
  identical: the PDF layout adds **line breaks** where the original text has spaces (for example,
  the name *Patricia Giménez Domínguez* or the address are split across two lines).
  This changes the character positions, so the gold spans (`start`, `end`) will not match the
  extracted text exactly, which will affect the **strict** evaluation in Task 2.
  A possible solution is to normalize whitespace (replace line breaks with spaces).

- **Scanned PDF:** the page contains only **one image** and PyMuPDF extracts **0 characters**,
  because a scanned document has no text layer: it is just a picture of the page.
  To get the text we need **OCR** (Optical Character Recognition) before applying GLiNER (Task 3).